# kodoom: ready-to-run Gemma 3 27B TPU translation

Import this configured notebook into a **private** Kaggle notebook. Enable
Internet, choose **TPU v5e-8**, enable the **HF_TOKEN** secret and attach the
version-1 Flax model described below. Then **Save Version > Save & Run All**.
No settings-cell edits are needed: **RUN_TPU=True**, your dataset and the tested
runtime pin are already configured.

Stable import/download: [execution.ipynb](https://raw.githubusercontent.com/Farahani1/kodum/codex/tpu-bulk-resume-batching/notebooks/execution.ipynb).

Checkpoint dataset: **Farahani/kodoom-kaggle-checkpoints**.
The early cell skips isolated CPU preparation. Short access/storage/model checks
run after production setup and must pass before translation. Source, token,
resume and real TPU/model checks remain in the worker.

Active data recipe: **bulk-translation**, project-plan **v21**, section 1.2;
configured production handoff: project-plan **v26**. Generate unreviewed drafts
of all **1,200 train + 400 test typed cases (8,000 decisions)** plus **2,000 helmo
records** with **gemma3-27b-tpu-bf16**, ending at
**drafts-complete-awaiting-review**.

Attach the model: open [Gemma 3 27B IT Flax version 1](https://www.kaggle.com/models/google/gemma-3/flax/gemma3-27b-it/1),
accept its terms, then use the notebook Input panel > Add Input / Add Models.
Select Gemma 3 > Flax > gemma3-27b-it > version 1. HF_TOKEN needs private dataset
read/write access; it does not grant Kaggle model access. Public GitHub code is
fetched anonymously. Never paste credentials into cells.

Keep the campaign and storage settings stable on resume. Changed code pins are
rejected for incompatible saved campaigns. An old unreleased worker must have
ended before choosing takeover. No automatic publication or human approval runs.

Optional CPU-only diagnosis: set RUN_TPU=False and choose Accelerator=None.
See docs/kaggle-huggingface-setup-guide.md and docs/execution-workflow.md.


In [ ]:
PROVIDER = "kaggle"
PROFILE = "kaggle-tpu"
BACKEND = "jax"
CODE_REVISION = "74e992e439d7bfb5f26e5d19ea8eea704769df6e"
CHECKOUT = "/tmp/kodoom-tpu-code"
CAMPAIGN_ID = "gemma27b-bulk-v1"
HF_DATASET_REPO = "Farahani/kodoom-kaggle-checkpoints"
OPERATOR = "owner"  # change this name for an authorized collaborator
RUN_TPU = True  # configured to launch translation on Kaggle TPU v5e-8
TAKEOVER = False  # True only after confirming an interrupted prior worker ended

## Access and setup before entering the TPU queue

With **RUN_TPU=False**, run this on CPU with Internet enabled. Check the exact
GitHub revision, HF token/private dataset read/write/readback, writable paths,
resume conflicts and attached version-1 Flax files. Only the HF client is added
to an isolated CPU environment. No corpus audit or TPU dependency dry-run runs.

A unique tiny private HF probe is written, read back and removed; its two commits
remain in repository history. No campaign writer lease is acquired.
Accept Gemma terms on Kaggle and attach the exact model under Inputs/Models.

With **RUN_TPU=True**, skip isolated CPU preparation. The next cell installs the
production dependencies and repeats short setup checks before translation.
Source/prompt/dependency audits are optional diagnostics described in the guide.


In [ ]:
import sys
import time
import types
import urllib.parse
import urllib.request

SESSION_STARTED = time.monotonic()
if not (3, 11) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("Use a Kaggle Linux runtime with Python 3.11 through 3.13")
if not HF_DATASET_REPO:
    raise ValueError("Set HF_DATASET_REPO before running CPU checks")


# Public source: do not send an optional stale GitHub credential.
url = (
    "https://raw.githubusercontent.com/Farahani1/kodum/"
    + urllib.parse.quote(CODE_REVISION, safe="")
    + "/src/kodoom/runtime.py"
)
with urllib.request.urlopen(url, timeout=30) as response:
    source = response.read().decode("utf-8")
runtime = types.ModuleType("kodoom_bootstrap")
exec(compile(source, "runtime.py", "exec"), runtime.__dict__)
# The pinned runtime also reads GitHub secrets during CPU and TPU checkout.
# Disable that credential here while preserving its HF secret adapter.
runtime.secret = lambda provider, name, read=runtime.secret: (
    None if name == "GITHUB_TOKEN" else read(provider, name)
)
preflight = None
if RUN_TPU:
    if not runtime.secret(PROVIDER, "HF_TOKEN"):
        raise ValueError("Enable HF_TOKEN for this notebook before TPU setup")
    print("TPU mode: isolated CPU preparation skipped; session setup checks run next.")
else:
    preflight = runtime.cpu_preflight(
        PROVIDER,
        CODE_REVISION,
        CHECKOUT,
        campaign_id=CAMPAIGN_ID,
        repo=HF_DATASET_REPO,
        operator=OPERATOR,
        takeover=TAKEOVER,
        profile=PROFILE,
        backend=BACKEND,
    )

## Explicit TPU opt-in: restore, benchmark and translate

This notebook is already configured with **RUN_TPU=True**. Select TPU v5e-8
and use **Save Version > Save & Run All**. The previous cell then
skips its isolated environment. This cell installs the pinned production
libraries, probes eight devices and checks this session's access, storage,
resume state and model attachment before starting the worker.

The worker validates source inputs, token limits, checkpoint identity and real
model loading. It benchmarks BF16 batches 1/2/4/8 and saves snapshots about every
ten minutes. The eight-hour budget starts in the early setup cell; 30 minutes
are reserved for finalization. With RUN_TPU=False this cell skips the TPU run.


In [ ]:
result = None
if RUN_TPU:
    checkout = runtime.bootstrap(PROVIDER, CODE_REVISION, CHECKOUT, backend=BACKEND)
    from kodoom.bulk.launch import run
    from kodoom.notebook_preflight import run_checks

    setup = run_checks(
        repo=HF_DATASET_REPO,
        campaign_id=CAMPAIGN_ID,
        operator=OPERATOR,
        revision=CODE_REVISION,
        takeover=TAKEOVER,
        report_path="/kaggle/working/kodoom/session-preflight.json",
    )
    if not setup["passed"]:
        raise RuntimeError("Session setup failed; fix the FAIL items before translation")

    result = run(
        provider=PROVIDER,
        profile=PROFILE,
        campaign_id=CAMPAIGN_ID,
        repo=HF_DATASET_REPO,
        operator=OPERATOR,
        takeover=TAKEOVER,
        session_started=SESSION_STARTED,
    )
else:
    print("CPU-only mode: TPU installation, model loading and translation skipped.")

## Progress and review

Remote `campaigns/<campaign-id>/checkpoint.json` identifies a verified archive
with English inputs, complete-case shards, progress, events, production settings,
TPU metrics and review CSVs. Local artifacts are under
`/kaggle/working/kodoom/data/bulk/<campaign-id>/`. Save private notebook outputs
as an additional copy. Keep model weights and caches out of saved output.

Review `review/cases-*.csv` and `review/labels-*.csv` with the English/Persian
context. Empty review fields remain unreviewed; automatic findings are flags.
All 400 test cases need human review. A diagnostic sample cannot establish
dataset-wide label error prevalence or detector coverage. Adoption, training
and publication still require the research decisions in the project plan.

Read-only snapshot download and pause instructions are in `docs/execution-workflow.md`.
Actual TPU fit, speed and a second live session remain unverified until run.


In [ ]:
if result is None:
    print("CPU preflight report:", "/kaggle/working/kodoom/cpu-preflight.json")
    print("After PASS: stop CPU session, select TPU v5e-8, set RUN_TPU=True, Save & Run All.")
else:
    print("Status:", result["status"])
    print("Completed / failed / total cases:", result["counts"])
    print("Private artifacts:", result["artifact_root"])
    print("Last verified remote revision:", result["last_verified_revision"])
    print("Final remote save verified:", result["persisted"])
    print("Stop here for human review; resume the same campaign if work remains.")